# 05 — Feature Engineering + Baseline Models (Phase 5)

โน้ตบุ๊กนี้เป็น narrative shell สำหรับ Phase 5: import และเรียกใช้ฟังก์ชันจาก `src/rri/features/`
และ `src/rri/models/` เท่านั้น — ไม่มี business logic ในเซลล์ ตาม CLAUDE.md N2

**กติกาที่ต้องยึด (CLAUDE.md N5/N6):**
- โมเดลเทรนจาก `fact_accident` (ข้อมูลจริง) เท่านั้น **ห้ามใช้ `fact_underwriting_monthly`**
  (พอร์ตจำลอง) — ไม่งั้นโมเดลจะย้อนกลับไปค้นพบสมมติฐานที่เราใส่เข้าไปเอง
- Time-based split เท่านั้น: train ≤2023, valid 2024, test 2025 — ห้าม random K-fold
  (`rri.features.splits.assert_time_based_split` เช็คใน code)

ลำดับขั้นตอน:

1. Feature engineering — สร้าง 2 modeling frame จาก warehouse
2. Severity Index correlation check (ค้างจาก Phase 4, CLAUDE.md §9)
3. Time-based split
4. GLM baselines (L0 — actuarial standard)
5. LightGBM baselines (L1)
6. SHAP explainability
7. ตารางเปรียบเทียบรวม
8. **Business-sense review addendum** (checkpoint decisions, 2026-08-24 — รายละเอียดเต็มที่
   `docs/phase5_modeling_findings.md`)

## 0. Setup

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import duckdb
import polars as pl
from sklearn.metrics import confusion_matrix, f1_score

from config.settings import settings
from rri.features.frequency_severity import build_frequency_severity_frame
from rri.features.severity_classification import build_severity_classification_frame
from rri.features.severity_index import (
    add_severity_index_coarse,
    add_severity_index_fine,
    severity_index_correlation,
)
from rri.features.splits import assert_time_based_split, time_based_split
from rri.models.glm_baseline import (
    compute_flood_fill_value,
    fit_gamma_severity_glm,
    fit_poisson_frequency_glm,
    frequency_training_rows,
    predict_glm,
    severity_training_rows,
)
from rri.models.lightgbm_baseline import (
    classifier_feature_matrix,
    fit_lightgbm_frequency,
    fit_lightgbm_severity,
    fit_lightgbm_severity_classifier,
    freq_sev_feature_matrix,
    predict_lightgbm_frequency,
    predict_lightgbm_severity,
    predict_lightgbm_severity_classifier,
)
from rri.models.metrics import (
    gamma_deviance,
    gini_coefficient,
    mean_absolute_error,
    poisson_deviance,
)
from rri.models.shap_explain import (
    compute_shap_values,
    multiclass_top_feature_directions,
    save_beeswarm,
    top_feature_directions,
)
from rri.utils.logging import configure_logging
from rri.warehouse.dim_vehicle import vehicle_group_mapping_path

configure_logging()

reports_dir = settings.reports_dir
reports_dir.mkdir(parents=True, exist_ok=True)

con = duckdb.connect(str(settings.warehouse_db_path), read_only=True)
fact_accident = con.sql("SELECT * FROM fact_accident").pl()
dim_date = con.sql("SELECT * FROM dim_date").pl()
dim_geography = con.sql("SELECT * FROM dim_geography").pl()
dim_vehicle = con.sql("SELECT * FROM dim_vehicle").pl()
dim_weather_band = con.sql("SELECT * FROM dim_weather_band").pl()
con.close()

rainfall_df = pl.read_parquet(settings.staging_dir / "rainfall.parquet")
stt_car_fuel_df = pl.read_parquet(settings.staging_dir / "stt_car_fuel.parquet")
car_registered_new_df = pl.read_parquet(settings.staging_dir / "car_registered_new.parquet")
accidents_staging = pl.read_parquet(settings.staging_dir / "accidents.parquet")
vehicle_mapping_path = vehicle_group_mapping_path()

print(
    "fact_accident:",
    fact_accident.height,
    "rows (CLAUDE.md N5: the only real-data source ML trains on)",
)

fact_accident: 159605 rows (CLAUDE.md N5: the only real-data source ML trains on)


## 1. Feature Engineering

**Frame 1 — ความถี่/ความรุนแรง (`frequency_severity`):** grain = จังหวัด × เดือน × กลุ่มรถ
หน้าต่างเวลา 2022-01 ถึง 2025-12 (48 เดือน) — ผูกกับช่วงที่ exposure back-cast มีข้อมูลจริง
(`car_registered_new` เริ่ม 2022-01) เหตุผลเดียวกับ `fact_underwriting_monthly` ใน Phase 4

**Frame 2 — จำแนกความรุนแรง (`severity_classification`):** grain = หนึ่งอุบัติเหตุ ใช้ categorical
ดิบ cardinality สูง (สาเหตุ, ลักษณะการเกิดเหตุ, ภูมิประเทศถนน, รหัสสายทาง) — frame นี้ที่
Phase 6 MLP + entity embeddings จะใช้ต่อ หน้าต่างเวลา 2020-2025 (ต้องมี injury breakdown)

In [2]:
freq_sev_frame = build_frequency_severity_frame(
    fact_accident,
    dim_date,
    dim_geography,
    dim_vehicle,
    rainfall_df,
    stt_car_fuel_df,
    car_registered_new_df,
    vehicle_mapping_path,
)
print("frequency_severity frame:", freq_sev_frame.shape)
print("zero-accident cells:", freq_sev_frame.filter(pl.col("accident_count") == 0).height)
freq_sev_frame.head(3)

frequency_severity frame: (18480, 38)
zero-accident cells: 4083


geography_sk,province_id,province_th,region,vehicle_sk,vehicle_group,has_normalized_exposure,year,month,season,accident_count,fatality_count,serious_injury_count,minor_injury_count,total_injury_count,severity_index_coarse,severity_index_fine,exposure_vehicle_count,log_exposure,rain_lead0,rain_lead1,rain_lead2,rain_lead3,rain_lead4,rain_lead5,flood_pct_high_risk_tambons,flood_peak_month,flood_data_available,r35mm_avg_days_2014_2023,ev_share,tourist_attraction_count,tourism_tier,n_holiday_days,n_long_weekend_days,has_songkran,has_newyear,avg_days_to_next_holiday,covid_disruption_level
i64,str,str,str,i64,str,bool,i64,i64,str,u32,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i32,bool,f64,f64,i64,str,u32,u32,bool,bool,f64,str
65,"""81""","""กระบี่""","""South""",4,"""heavy_truck_bus""",false,2022,1,"""cool""",1,0,0,0,0,0.0,0.0,null,null,33.960728,10.955149,227.446833,147.228675,221.076067,130.723328,0.0,null,true,14.4,0.013755,25,"""high""",2,3,false,true,26.677419,"""none"""
65,"""81""","""กระบี่""","""South""",4,"""heavy_truck_bus""",false,2022,2,"""cool""",1,0,0,1,1,2.15,1.0,null,null,10.955149,227.446833,147.228675,221.076067,130.723328,182.847974,0.0,null,true,14.4,0.013755,25,"""high""",1,0,false,false,22.5,"""none"""
65,"""81""","""กระบี่""","""South""",4,"""heavy_truck_bus""",false,2022,3,"""hot""",1,0,0,0,0,0.0,0.0,null,null,227.446833,147.228675,221.076067,130.723328,182.847974,211.491251,0.0,null,true,14.4,0.013755,25,"""high""",0,0,false,false,21.0,"""none"""


In [3]:
sev_class_frame = build_severity_classification_frame(
    accidents_staging, fact_accident, dim_date, dim_geography, dim_vehicle, dim_weather_band
)
print("severity_classification frame:", sev_class_frame.shape)
sev_class_frame["severity_class"].value_counts().sort("severity_class")

severity_classification frame: (131672, 31)


severity_class,count
str,u32
"""fatal""",14763
"""minor""",48696
"""none""",58238
"""serious""",9975


### Severity Index correlation check (ค้างจาก Phase 4, CLAUDE.md §9)

`severity_index_coarse` (ใช้ได้ทุกปี) กับ `severity_index_fine` (ใช้ได้เฉพาะ 2020-2025
ที่มี injury breakdown) ควรสัมพันธ์กันสูง — ถ้าไม่ ก็เป็นสัญญาณว่า resolution ที่เสียไปจากการใช้
coarse index มีนัยสำคัญ

In [4]:
fact_with_year = fact_accident.join(dim_date.select("date_sk", "year"), on="date_sk", how="left")
fact_with_year = add_severity_index_coarse(fact_with_year)
fact_with_year = add_severity_index_fine(fact_with_year)
correlation_result = severity_index_correlation(fact_with_year)
year_range = f"{correlation_result['year_start']}-{correlation_result['year_end']}"
r_value = correlation_result["pearson_r"]
n_rows = correlation_result["n_rows"]
print(f"Pearson r (coarse vs. fine, {year_range}): {r_value:.4f} (n={n_rows:,})")

Pearson r (coarse vs. fine, 2020-2025): 0.9816 (n=131,672)


## 2. Time-based split (CLAUDE.md N6)

In [5]:
freq_sev_split = time_based_split(freq_sev_frame)
sev_class_split = time_based_split(sev_class_frame)

splits_to_check = [
    ("frequency_severity", freq_sev_split),
    ("severity_classification", sev_class_split),
]
for name, split in splits_to_check:
    assert_time_based_split(split)  # re-asserted, though time_based_split already does it
    print(
        f"{name}: train={split.train.height} (<=2023) "
        f"valid={split.valid.height} (2024) test={split.test.height} (2025)"
    )

frequency_severity: train=9240 (<=2023) valid=4620 (2024) test=4620 (2025)
severity_classification: train=84005 (<=2023) valid=23952 (2024) test=23715 (2025)


## 3. GLM Baselines (L0 — actuarial standard)

Poisson GLM (offset = `log_exposure`) สำหรับความถี่ · Gamma GLM สำหรับความรุนแรง
(เฉพาะ cell ที่ `severity_index_coarse > 0` — Gamma ต้องการค่าเป็นบวกเท่านั้น)

In [6]:
freq_train = frequency_training_rows(freq_sev_split.train)
freq_valid = frequency_training_rows(freq_sev_split.valid)
freq_test = frequency_training_rows(freq_sev_split.test)
flood_fill = compute_flood_fill_value(freq_train)

glm_freq = fit_poisson_frequency_glm(freq_train, flood_fill)
glm_freq_pred_test = predict_glm(glm_freq, freq_test, flood_fill)

glm_freq_deviance = poisson_deviance(freq_test["accident_count"].to_numpy(), glm_freq_pred_test)
glm_freq_gini = gini_coefficient(freq_test["accident_count"].to_numpy(), glm_freq_pred_test)
print(f"GLM frequency -- Poisson deviance (test): {glm_freq_deviance:.3f}")
print(f"GLM frequency -- Gini (test): {glm_freq_gini:.3f}")

GLM frequency -- Poisson deviance (test): 16.666
GLM frequency -- Gini (test): 0.751


In [7]:
sev_train = severity_training_rows(freq_sev_split.train)
sev_valid = severity_training_rows(freq_sev_split.valid)
sev_test = severity_training_rows(freq_sev_split.test)

glm_sev = fit_gamma_severity_glm(sev_train, flood_fill)
glm_sev_pred_test = predict_glm(glm_sev, sev_test, flood_fill)

glm_sev_deviance = gamma_deviance(sev_test["severity_index_coarse"].to_numpy(), glm_sev_pred_test)
glm_sev_mae = mean_absolute_error(sev_test["severity_index_coarse"].to_numpy(), glm_sev_pred_test)
glm_sev_gini = gini_coefficient(sev_test["severity_index_coarse"].to_numpy(), glm_sev_pred_test)
print(f"GLM severity -- Gamma deviance (test): {glm_sev_deviance:.3f}")
print(f"GLM severity -- MAE (test): {glm_sev_mae:.3f}")
print(f"GLM severity -- Gini (test): {glm_sev_gini:.3f}")

GLM severity -- Gamma deviance (test): 1.213
GLM severity -- MAE (test): 8.499
GLM severity -- Gini (test): 0.499


## 4. LightGBM Baselines (L1)

เช่นเดียวกับ GLM: `objective=poisson`/`gamma` แยกความถี่/ความรุนแรง บวก classifier แยก
สำหรับ severity class (4 ระดับ) บน record-level frame

**หมายเหตุ:** `avg_days_to_next_holiday` ถูกตัดออกจาก feature set ของ LightGBM
frequency/severity แล้ว (checkpoint 2026-08-24 — partial-dependence ไม่ coherent, ดู
รายละเอียดในหัวข้อ 8 ด้านล่าง และ `docs/phase5_modeling_findings.md`)

In [8]:
lgbm_freq = fit_lightgbm_frequency(freq_train, freq_valid)
lgbm_freq_pred_test = predict_lightgbm_frequency(lgbm_freq, freq_test)

lgbm_freq_deviance = poisson_deviance(freq_test["accident_count"].to_numpy(), lgbm_freq_pred_test)
lgbm_freq_gini = gini_coefficient(freq_test["accident_count"].to_numpy(), lgbm_freq_pred_test)
best_iter = lgbm_freq.booster.best_iteration
print(
    f"LightGBM frequency -- Poisson deviance (test): {lgbm_freq_deviance:.3f} "
    f"(best_iteration={best_iter})"
)
print(f"LightGBM frequency -- Gini (test): {lgbm_freq_gini:.3f}")

LightGBM frequency -- Poisson deviance (test): 27.540 (best_iteration=445)
LightGBM frequency -- Gini (test): 0.773


In [9]:
lgbm_sev = fit_lightgbm_severity(sev_train, sev_valid)
lgbm_sev_pred_test = predict_lightgbm_severity(lgbm_sev, sev_test)

lgbm_sev_deviance = gamma_deviance(sev_test["severity_index_coarse"].to_numpy(), lgbm_sev_pred_test)
lgbm_sev_mae = mean_absolute_error(sev_test["severity_index_coarse"].to_numpy(), lgbm_sev_pred_test)
lgbm_sev_gini = gini_coefficient(sev_test["severity_index_coarse"].to_numpy(), lgbm_sev_pred_test)
print(f"LightGBM severity -- Gamma deviance (test): {lgbm_sev_deviance:.3f}")
print(f"LightGBM severity -- MAE (test): {lgbm_sev_mae:.3f}")
print(f"LightGBM severity -- Gini (test): {lgbm_sev_gini:.3f}")

/Users/phidphew/Documents/KU Work/Pee 3/Data Mining/group_project/project/.venv/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


LightGBM severity -- Gamma deviance (test): 1.209
LightGBM severity -- MAE (test): 8.241
LightGBM severity -- Gini (test): 0.511


In [10]:
lgbm_clf = fit_lightgbm_severity_classifier(sev_class_split.train, sev_class_split.valid)
clf_pred_test = predict_lightgbm_severity_classifier(lgbm_clf, sev_class_split.test)
clf_y_test = sev_class_split.test["severity_class"].to_numpy()

macro_f1 = f1_score(clf_y_test, clf_pred_test, average="macro")
cm = confusion_matrix(clf_y_test, clf_pred_test, labels=lgbm_clf.model.classes_)
print(f"LightGBM severity classifier -- macro-F1 (test): {macro_f1:.3f}")
print("classes:", list(lgbm_clf.model.classes_))
print("confusion matrix (rows=actual, cols=predicted):")
print(cm)

/Users/phidphew/Documents/KU Work/Pee 3/Data Mining/group_project/project/.venv/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


LightGBM severity classifier -- macro-F1 (test): 0.398
classes: ['fatal', 'minor', 'none', 'serious']
confusion matrix (rows=actual, cols=predicted):
[[ 803  986  552   46]
 [ 471 3723 4616   99]
 [ 118 2123 8527   26]
 [ 232  818  520   55]]


## 5. SHAP Explainability

`shap.TreeExplainer` บน test set — บันทึก beeswarm (PNG) + ตาราง top-8 feature ลง `reports/`

In [11]:
freq_x_test = freq_sev_feature_matrix(freq_test, lgbm_freq.levels)
freq_shap = compute_shap_values(lgbm_freq.booster, freq_x_test)
freq_shap_table = top_feature_directions(freq_shap, n=8)
save_beeswarm(freq_shap, reports_dir / "shap_beeswarm_frequency.png")
freq_shap_table.write_csv(reports_dir / "shap_top_features_frequency.csv")
freq_shap_table

feature,mean_abs_shap,value_shap_correlation,direction
str,f64,f64,str
"""vehicle_group""",0.830794,0.952917,"""increases"""
"""ev_share""",0.206157,-0.827244,"""decreases"""
"""region""",0.091813,-0.355895,"""decreases"""
"""rain_lead5""",0.088237,0.665113,"""increases"""
"""n_long_weekend_days""",0.082143,-0.144602,"""decreases"""
"""r35mm_avg_days_2014_2023""",0.07478,-0.482418,"""decreases"""
"""rain_lead0""",0.074042,-0.365318,"""decreases"""
"""rain_lead4""",0.057926,0.654098,"""increases"""


In [12]:
sev_x_test = freq_sev_feature_matrix(sev_test, lgbm_sev.levels)
sev_shap = compute_shap_values(lgbm_sev.model, sev_x_test)
sev_shap_table = top_feature_directions(sev_shap, n=8)
save_beeswarm(sev_shap, reports_dir / "shap_beeswarm_severity.png")
sev_shap_table.write_csv(reports_dir / "shap_top_features_severity.csv")
sev_shap_table

feature,mean_abs_shap,value_shap_correlation,direction
str,f64,f64,str
"""vehicle_group""",0.353438,-0.518589,"""decreases"""
"""ev_share""",0.095606,-0.601071,"""decreases"""
"""region""",0.066307,0.183213,"""increases"""
"""flood_pct_high_risk_tambons""",0.027088,0.283111,"""increases"""
"""tourism_tier""",0.023848,0.787951,"""increases"""
"""has_newyear""",0.020532,-0.710901,"""decreases"""
"""r35mm_avg_days_2014_2023""",0.019514,0.296261,"""increases"""
"""rain_lead5""",0.019153,0.107551,"""increases"""


In [13]:
# Classifier: SHAP sliced to the "fatal" class -- the most business-critical
# severity tier (CLAUDE.md §10 L4 uses this for insight narrative later).
clf_sample = sev_class_split.test.sample(n=3000, seed=42)
clf_x_test = classifier_feature_matrix(clf_sample, lgbm_clf.levels)
clf_shap = compute_shap_values(lgbm_clf.model, clf_x_test)
class_names = list(lgbm_clf.model.classes_)
clf_shap_table = multiclass_top_feature_directions(clf_shap, class_names, "fatal", n=8)
clf_shap_table.write_csv(reports_dir / "shap_top_features_classifier_fatal.csv")
clf_shap_table

feature,mean_abs_shap,value_shap_correlation,direction
str,f64,f64,str
"""road_code""",0.547873,0.320051,"""increases"""
"""crash_type_raw""",0.424468,0.249917,"""increases"""
"""vehicle_group""",0.292456,0.043779,"""mixed/unclear"""
"""cause_raw""",0.093395,-0.109397,"""decreases"""
"""ev_share""",0.074081,-0.913753,"""decreases"""
"""road_geometry_raw""",0.048183,0.224931,"""increases"""
"""region""",0.045976,-0.501503,"""decreases"""
"""weather_band""",0.030257,-0.479719,"""decreases"""


## 6. ตารางเปรียบเทียบรวม (GLM vs. LightGBM) → `reports/`

In [14]:
comparison = pl.DataFrame(
    {
        "model": ["GLM (Poisson)", "LightGBM (Poisson)", "GLM (Gamma)", "LightGBM (Gamma)"],
        "target": ["frequency", "frequency", "severity", "severity"],
        "deviance": [glm_freq_deviance, lgbm_freq_deviance, glm_sev_deviance, lgbm_sev_deviance],
        "gini": [glm_freq_gini, lgbm_freq_gini, glm_sev_gini, lgbm_sev_gini],
    }
)
comparison.write_csv(reports_dir / "phase5_model_comparison.csv")
comparison

model,target,deviance,gini
str,str,f64,f64
"""GLM (Poisson)""","""frequency""",16.665911,0.751378
"""LightGBM (Poisson)""","""frequency""",27.54029,0.772877
"""GLM (Gamma)""","""severity""",1.213328,0.498792
"""LightGBM (Gamma)""","""severity""",1.208744,0.511317


## 7. Business-Sense Review Addendum (checkpoint decisions, 2026-08-24)

รายละเอียดเต็มอยู่ที่ `docs/phase5_modeling_findings.md` — ที่นี่แสดงเฉพาะตัวเลขหลักที่ใช้ตัดสินใจ

### 7.1 `ev_share` เป็น confound ไม่ใช่ effect เชิงสาเหตุ (สำหรับ BQ8)

SHAP magnitude ของ `ev_share` สูงผิดปกติในทุกโมเดล (GLM coefficient −40.8) — ตรวจสอบว่า
เป็น proxy ของความเป็นเมือง/ขนาดฝูงยานพาหนะ ไม่ใช่ผล EV จริง:

In [15]:
from rri.cleaning.province import normalize_province

dg = dim_geography.filter(pl.col("geography_sk") != -1).select("province_th", "region", "ev_share")
vehicle_totals = (
    stt_car_fuel_df.with_columns(
        pl.col("จำนวนรถ").cast(pl.Int64, strict=False).fill_null(0).alias("_count"),
        pl.col("จังหวัด").map_elements(normalize_province, return_dtype=pl.Utf8).alias("province_th"),
    )
    .filter(pl.col("province_th").is_not_null())
    .group_by("province_th")
    .agg(pl.col("_count").sum().alias("total_vehicles"))
)
ev_diagnostic = dg.join(vehicle_totals, on="province_th", how="left")

corr_raw = ev_diagnostic.select(pl.corr("ev_share", "total_vehicles")).item()
corr_log = ev_diagnostic.select(pl.corr("ev_share", pl.col("total_vehicles").log())).item()
print(f"Corr(ev_share, total registered vehicles): r={corr_raw:.3f}")
print(f"Corr(ev_share, log(total registered vehicles)): r={corr_log:.3f}")
print()
print("Top 5 provinces by ev_share:")
print(ev_diagnostic.sort("ev_share", descending=True).head(5))
ev_diagnostic.write_csv(reports_dir / "ev_share_confound_diagnostic.csv")

Corr(ev_share, total registered vehicles): r=0.829
Corr(ev_share, log(total registered vehicles)): r=0.718

Top 5 provinces by ev_share:
shape: (5, 4)
┌──────────────┬─────────┬──────────┬────────────────┐
│ province_th  ┆ region  ┆ ev_share ┆ total_vehicles │
│ ---          ┆ ---     ┆ ---      ┆ ---            │
│ str          ┆ str     ┆ f64      ┆ i64            │
╞══════════════╪═════════╪══════════╪════════════════╡
│ กรุงเทพมหานคร ┆ Central ┆ 0.062281 ┆ 12372559       │
│ ภูเก็ต         ┆ South   ┆ 0.030332 ┆ 570419         │
│ เชียงใหม่      ┆ North   ┆ 0.022548 ┆ 1657767        │
│ สงขลา        ┆ South   ┆ 0.022248 ┆ 946788         │
│ นนทบุรี        ┆ Central ┆ 0.020211 ┆ 164318         │
└──────────────┴─────────┴──────────┴────────────────┘


**สรุป:** `ev_share` กระจุกตัวในจังหวัดเมืองใหญ่/ท่องเที่ยว (กรุงเทพฯ, ภูเก็ต, เชียงใหม่,
สงขลา) และสัมพันธ์แน่นกับขนาดฝูงยานพาหนะทั้งจังหวัด (r≈0.72-0.83) — เป็นลายเซ็นของ
province-identity proxy ไม่ใช่ EV effect จริง **การตัดสินใจ:** เก็บ feature ไว้ในโมเดล
พยากรณ์ (ใช้งานได้จริง ไม่ leak) แต่ **ห้ามใช้ตอบ BQ8** — BQ8 ต้องตอบผ่านข้อจำกัดที่ระบุไว้
เท่านั้น: `fact_accident` ไม่มี fuel type ต่อคัน เปรียบเทียบได้แค่ระดับจังหวัด (ecological)
ไม่ใช่ต่อคัน

### 7.2 `avg_days_to_next_holiday` — ตัดออกจาก feature set แล้ว (ดูหัวข้อ 4 ด้านบน)

Partial-dependence ไม่ coherent (flat → spike → decay) และสัมพันธ์ −0.71 กับ
`n_holiday_days` ที่ signal สะอาดกว่าอยู่แล้ว — รายละเอียดเต็มใน
`docs/phase5_modeling_findings.md` §2

### 7.3 `r35mm_avg_days_2014_2023` สลับเครื่องหมายระหว่างความถี่/ความรุนแรง — insight สำหรับ BQ2

จากตาราง SHAP หัวข้อ 5: `r35mm_avg_days_2014_2023` เป็น "decreases" ในโมเดลความถี่ แต่เป็น
"increases" ในโมเดลความรุนแรง — ไม่ใช่ความขัดแย้ง แต่เป็นภาพที่สอดคล้องกัน: **จังหวัดฝนตกหนัก
บ่อยมีอุบัติเหตุน้อยกว่า (การจราจรเบาบางกว่า) แต่อุบัติเหตุที่เกิดรุนแรงกว่า** (ถนนความเร็วสูง
ชนบท + สภาพอากาศเสี่ยง) สอดคล้องกับ `flood_pct_high_risk_tambons` ที่เป็น severity driver
บวกเช่นกัน (คนละแหล่งข้อมูล แต่ชี้ทิศทางเดียวกัน) — ใช้เป็น headline finding สำหรับ BQ2

### 7.4 `rain_lead0..5` มี collinearity สูง — ต้องเช็คใน Phase 6 GRU

`rain_lead4`/`rain_lead5` (พยากรณ์ล่วงหน้า 4-5 เดือน) มี SHAP importance สูงกว่า `rain_lead0`
(พยากรณ์ใกล้สุด) ในทั้งสองโมเดล — จุดขายหลักของโปรเจกต์คือ "พยากรณ์ฝนล่วงหน้าทำนายอนาคตได้"
ถ้า 6 คอลัมน์นี้ทำหน้าที่แค่เป็น proxy ของ "จังหวัดนี้ฝนตกมาก/น้อย" โดยรวม ไม่ได้ให้สัญญาณ
เฉพาะเดือนจริง ความแตกต่างของ forecast lead ก็มีค่าน้อยกว่าที่คิด — **Phase 6 ต้องตรวจสอบว่า
GRU ใช้ lead ต่างกันจริงสำหรับเดือนต่างกันหรือไม่** และรายงานผลตามจริงไม่ว่าจะเป็นด้านไหน

## CHECKPOINT — Business-Sense Review ผ่านแล้ว (2026-08-24)

สรุปสำหรับผู้อ่านที่ตรวจทาน: metric table + SHAP top-8 (หัวข้อ 5-6), severity index
correlation (หัวข้อ 1), และประเด็นที่ต้องระวัง 4 ข้อ (หัวข้อ 7) ถูกทบทวนและตัดสินใจแล้ว
โมเดล baseline นี้เป็นฐานที่มั่นคงสำหรับ Phase 6 (neural net + forecast)

In [16]:
print("=== GLM vs LightGBM (test 2025) ===")
print(comparison)
print()
final_r = correlation_result["pearson_r"]
print(f"severity_index_coarse vs. fine correlation (2020-2025): r={final_r:.4f}")
print()
print("=== Top 8 SHAP features -- frequency ===")
print(freq_shap_table)
print()
print("=== Top 8 SHAP features -- severity ===")
print(sev_shap_table)
print()
print(f"ev_share vs. total vehicles correlation: r={corr_raw:.3f} (raw), r={corr_log:.3f} (log)")

=== GLM vs LightGBM (test 2025) ===
shape: (4, 4)
┌────────────────────┬───────────┬───────────┬──────────┐
│ model              ┆ target    ┆ deviance  ┆ gini     │
│ ---                ┆ ---       ┆ ---       ┆ ---      │
│ str                ┆ str       ┆ f64       ┆ f64      │
╞════════════════════╪═══════════╪═══════════╪══════════╡
│ GLM (Poisson)      ┆ frequency ┆ 16.665911 ┆ 0.751378 │
│ LightGBM (Poisson) ┆ frequency ┆ 27.54029  ┆ 0.772877 │
│ GLM (Gamma)        ┆ severity  ┆ 1.213328  ┆ 0.498792 │
│ LightGBM (Gamma)   ┆ severity  ┆ 1.208744  ┆ 0.511317 │
└────────────────────┴───────────┴───────────┴──────────┘

severity_index_coarse vs. fine correlation (2020-2025): r=0.9816

=== Top 8 SHAP features -- frequency ===
shape: (8, 4)
┌──────────────────────────┬───────────────┬────────────────────────┬───────────┐
│ feature                  ┆ mean_abs_shap ┆ value_shap_correlation ┆ direction │
│ ---                      ┆ ---           ┆ ---                    ┆ ---       │
│ 